# MevzuatRadar — açık kaynak LLM ile değişiklik çıkarımı

Bu defter, **eğitim yapmadan**, yalnızca talimat ve birkaç örnekle (few-shot) bir açık kaynak dil modelini
çalıştırır. İstemler projede `mevzuatradar export-llm` ile üretilir; defter sadece modeli koşturur.

Çıktı biçimi mT5 modeliyle aynıdır, böylece kural sistemi / mT5 / LLM üçü de aynı doğrulama hattından geçer.

**Başlamadan önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.*

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers>=4.46" accelerate bitsandbytes

## 1. İstem dosyalarını yükle
Bilgisayarındaki `data/ml/llm_dev.jsonl` ve `data/ml/llm_test.jsonl` dosyalarını birlikte seç.

In [ ]:
import json
from google.colab import files

yuklenen = files.upload()

def oku(yol):
    return [json.loads(l) for l in open(yol, encoding="utf-8")]

dev = oku("llm_dev.jsonl")
test = oku("llm_test.jsonl")
print(f"geliştirme {len(dev)} istem | test {len(test)} istem")
print("\nÖrnek istemin son mesajı:\n", dev[0]["messages"][-1]["content"][:300])
print("\nTalimatın ilk satırları:\n", "\n".join(dev[0]["messages"][0]["content"].split("\n")[:6]))

## 2. Modeli yükle (4-bit)
T4'e sığması için model 4-bit niceleme ile yüklenir (~5 GB). İndirme birkaç dakika sürer.
Bellek yetmezse `MODEL` satırını daha küçük bir modelle değiştir (ör. `Qwen/Qwen2.5-3B-Instruct`).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-7B-Instruct"   # alternatif: "Qwen/Qwen2.5-3B-Instruct"

nicele = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=nicele, device_map="auto")
model.eval()
print("yüklendi:", MODEL)

## 3. Tahmin üret
Örneklemesiz (deterministik) üretim: aynı istem her zaman aynı cevabı verir.
145 madde için T4'te yaklaşık 20-40 dakika sürer.

In [ ]:
import re, time

def temizle(metin):
    """Model bazen kod bloğu veya açıklama ekler; kayıt satırlarını ayıklar."""
    metin = re.sub(r"```[a-zA-Z]*", "", metin).replace("```", "").strip()
    satirlar = [s.strip() for s in metin.split("\n") if s.strip()]
    kayitlar = [s for s in satirlar if s == "YOK" or re.match(r"^[A-Z_]+\s*\|", s)]
    return " ;; ".join(kayitlar) if kayitlar else (satirlar[0] if satirlar else "")

def uret(istemler, dosya, max_yeni=320):
    sonuc, baslangic = [], time.time()
    for i, ornek in enumerate(istemler):
        metin = tokenizer.apply_chat_template(ornek["messages"], tokenize=False, add_generation_prompt=True)
        enc = tokenizer(metin, return_tensors="pt").to(model.device)
        with torch.no_grad():
            cikti = model.generate(**enc, max_new_tokens=max_yeni, do_sample=False,
                                   pad_token_id=tokenizer.eos_token_id)
        cevap = tokenizer.decode(cikti[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
        sonuc.append({"id": ornek["id"], "prediction": temizle(cevap)})
        if (i + 1) % 10 == 0 or i + 1 == len(istemler):
            gecen = time.time() - baslangic
            print(f"  {i + 1}/{len(istemler)} | {gecen / (i + 1):.1f} sn/madde | kalan ~{gecen / (i + 1) * (len(istemler) - i - 1) / 60:.0f} dk")
    with open(dosya, "w", encoding="utf-8") as f:
        for p in sonuc:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(dosya, "yazıldı")
    return sonuc

p_dev = uret(dev, "preds_llm_dev.jsonl")
p_test = uret(test, "preds_llm_test.jsonl")

for ornek, tahmin in list(zip(test, p_test))[:5]:
    print("\nGİRDİ :", ornek["messages"][-1]["content"][:200])
    print("MODEL :", tahmin["prediction"][:300])

In [ ]:
files.download("preds_llm_dev.jsonl")
files.download("preds_llm_test.jsonl")

## 4. Sonra
İndirilen dosyaları projede `data/ml/` klasörüne koyup bilgisayarda çalıştır:
```
mevzuatradar evaluate-model --pred data/ml/preds_llm_dev.jsonl --split dev
mevzuatradar evaluate-model --pred data/ml/preds_llm_test.jsonl --split test
```